# The replies

What came back, before any of it is scored. Two things are worth checking here
rather than after the metrics: how many replies could not be read at all, and
how many declined rather than judged.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

In [ ]:
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
%load_ext autoreload
%autoreload 2

import backends
import evaluate
import prompts as templates
import run
import settings
import utils

utils.make_directories()
pd.set_option('display.max_colwidth', 90)
# Built from the dataset rather than shipped, so that what the pipeline reads is
# always derived from what is in data/halomi rather than from a stale copy.
if not settings.ITEMS_PATH.exists():
    raise SystemExit(
        'Nothing has been built yet. From the repository root, run:\n'
        '    python scripts/build.py\n'
        'That writes data/benchmark/items.csv and prompts.csv, which every '
        'notebook and stage reads.')

print('Ready')

## Coverage

In [ ]:
rows = []
for entry in settings.MODELS.values():
    for method in settings.METHODS:
        replies = utils.read_lines(utils.response_path(entry['id'], method))
        if replies.empty:
            continue
        rows.append({'model': utils.model_slug(entry['id']), 'method': method,
                     'replies': len(replies),
                     'truncated': int(replies.get('truncated', pd.Series(dtype=bool)).sum()),
                     'errored': int((replies['error'].astype(str).str.strip() != '').sum())})
display(pd.DataFrame(rows) if rows else 'Nothing collected yet')

## Unreadable and refused

A reply that gives no label and a reply that declines are different failures. The
first is a parsing problem, the second is a decision by the model, and the three
views in evaluate.py exist to keep them apart.

In [ ]:
MODEL = list(settings.MODELS.values())[0]['id']
judged = evaluate.judge(MODEL, 'baseline')

if judged.empty:
    print('Nothing collected for this model yet')
else:
    print(f"{len(judged):,} replies")
    print(f"  {int((~judged['parsed']).sum()):>5} gave no label")
    print(f"  {int(judged['refusal'].sum()):>5} declined to judge")
    display(pd.crosstab(judged['direction'], judged['refusal']))

## Reading a few

In [ ]:
replies = utils.read_lines(utils.response_path(MODEL, 'baseline'))
if not replies.empty:
    prompts = utils.read_table(settings.PROMPTS_PATH)
    merged = replies.merge(prompts[['prompt_id', 'direction', 'answer']],
                           on='prompt_id')
    display(merged[['direction', 'answer', 'response']].head(5))